# Stage 1 — Pretraining (STS-UVD on BraTS2021)
**Project:** ContextualDenoising-BrainTumorSSL  
**Goal:** Self-supervised denoising pretraining → save encoder weights for transfer  

### Session checklist
- [ ] GPU accelerator enabled (Settings → Accelerator → GPU T4 x2)
- [ ] Internet ON (Settings → Internet → On)  ← needed for git clone
- [ ] Datasets attached: `ashfakazadnafi/brats2021-tar`

## 0. Install dependencies

In [ ]:
%%capture
!pip install monai nibabel scikit-image scipy pyyaml tqdm matplotlib

## 1. Clone your GitHub repo

In [ ]:
import os

REPO_URL  = 'https://github.com/Nafi14078/ContextualDenoising-BrainTumorSSL.git'
REPO_NAME = 'ContextualDenoising-BrainTumorSSL'
PROJECT   = 'BrainTumor_SSL_Project'

# Clone fresh every session (runtime resets wipe /kaggle/working)
if os.path.exists(f'/kaggle/working/{REPO_NAME}'):
    !rm -rf /kaggle/working/{REPO_NAME}

!git clone {REPO_URL} /kaggle/working/{REPO_NAME}

# Add project root to Python path
PROJECT_ROOT = f'/kaggle/working/{REPO_NAME}/{PROJECT}'
import sys
sys.path.insert(0, PROJECT_ROOT)

print(f'✓ Repo cloned')
print(f'  Project root: {PROJECT_ROOT}')
!ls {PROJECT_ROOT}

## 2. Verify dataset paths

In [ ]:
import os

# ── BraTS2021 root (parent of all BraTS2021_XXXXX folders) ──
BRATS2021_ROOT = '/kaggle/input/datasets/ashfakazadnafi/brats2021-tar/brats2021/datasets/brats2021'

# ── Output directory for preprocessed slices ──
SLICES_OUT = '/kaggle/working/brats2021_slices'

# ── Checkpoint directory ──
CKPT_DIR = '/kaggle/working/checkpoints/pretrain'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(SLICES_OUT, exist_ok=True)

# Verify BraTS2021 root exists and list first 3 subjects
assert os.path.exists(BRATS2021_ROOT), f'NOT FOUND: {BRATS2021_ROOT}'
subjects = sorted(os.listdir(BRATS2021_ROOT))
print(f'✓ BraTS2021 root found')
print(f'  Total subjects : {len(subjects)}')
print(f'  First 3        : {subjects[:3]}')

# Verify one subject has all 4 modalities
sample = os.path.join(BRATS2021_ROOT, subjects[0])
files  = os.listdir(sample)
print(f'\n  Sample subject files:')
for f in sorted(files): print(f'    {f}')

## 3. Preprocess BraTS2021 → 2D axial slices
**Run once.** After this cell finishes, upload `brats2021_slices/` as a Kaggle Dataset called `brats2021-preprocessed` so you don't redo this every session.

In [ ]:
# Check if already preprocessed (skip if re-running session)
import os, json
meta_path = f'{SLICES_OUT}/metadata.json'

if os.path.exists(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)
    print(f'✓ Already preprocessed — skipping')
    print(f'  Train slices: {len(meta["train"])}')
    print(f'  Val slices  : {len(meta["val"])}')
else:
    print('Running preprocessing...')
    !python {PROJECT_ROOT}/scripts/preprocess_brats2021.py \
        --root {BRATS2021_ROOT} \
        --out  {SLICES_OUT} \
        --skip 10 \
        --min_brain 0.05

## 4. Update config with correct paths

In [ ]:
import yaml

cfg_path = f'{PROJECT_ROOT}/configs/pretrain_config.yaml'

with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# ── Override paths with Kaggle-specific values ──
cfg['data']['brats2021_root']  = BRATS2021_ROOT
cfg['data']['output_slices']   = SLICES_OUT
cfg['training']['checkpoint_dir'] = CKPT_DIR

# ── Save updated config ──
updated_cfg_path = '/kaggle/working/pretrain_config_kaggle.yaml'
with open(updated_cfg_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('✓ Config updated')
print(f'  output_slices   : {cfg["data"]["output_slices"]}')
print(f'  checkpoint_dir  : {cfg["training"]["checkpoint_dir"]}')
print(f'  epochs          : {cfg["training"]["epochs"]}')
print(f'  batch_size      : {cfg["training"]["batch_size"]}')
print(f'  STS N (window)  : {cfg["sts"]["N"]}')

## 5. Sanity check — visualise a slice window

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

train_dir = f'{SLICES_OUT}/train'
sample_file = sorted(os.listdir(train_dir))[100]   # pick slice #100
sample = np.load(f'{train_dir}/{sample_file}')      # (4, H, W)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
modalities = ['T1', 'T1ce', 'T2', 'FLAIR']
for i, ax in enumerate(axes):
    ax.imshow(sample[i], cmap='gray')
    ax.set_title(modalities[i])
    ax.axis('off')
plt.suptitle(f'Sample slice: {sample_file}', fontsize=12)
plt.tight_layout()
plt.show()
print(f'Shape: {sample.shape}  |  Min: {sample.min():.3f}  |  Max: {sample.max():.3f}')

## 6. Run Pretraining
⚠️ **Kaggle session = ~9 hours.** Checkpoints are saved every 5 epochs to `/kaggle/working/checkpoints/pretrain/`.  
After the session ends, **download `best_model.pth` and `pretrain_encoder.pth`** immediately.

In [ ]:
# Optional: resume from a previous checkpoint
# Upload your previous checkpoint as a Kaggle Dataset and set path here
RESUME_FROM = None   # e.g. '/kaggle/input/pretrain-ckpt/epoch_010.pth'

import subprocess, sys

cmd = [
    sys.executable,
    f'{PROJECT_ROOT}/pretraining/train_pretrain.py',
    '--config', updated_cfg_path
]
if RESUME_FROM:
    cmd += ['--resume', RESUME_FROM]

# Run training (output streams live to notebook)
process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in process.stdout:
    print(line, end='')
process.wait()
print(f'\nProcess exited with code: {process.returncode}')

## 7. Verify outputs + show PSNR/SSIM

In [ ]:
import os, torch

encoder_path = f'{CKPT_DIR}/pretrain_encoder.pth'
best_path    = f'{CKPT_DIR}/best_model.pth'

for path in [encoder_path, best_path]:
    if os.path.exists(path):
        size = os.path.getsize(path) / 1e6
        print(f'✓ {os.path.basename(path)}  ({size:.1f} MB)')
    else:
        print(f'✗ NOT FOUND: {path}')

# Load best checkpoint and print metrics
if os.path.exists(best_path):
    ckpt = torch.load(best_path, map_location='cpu')
    print(f'\n  Best epoch : {ckpt["epoch"] + 1}')
    print(f'  Best PSNR  : {ckpt.get("psnr", "N/A")}')
    print(f'\n⚠️  Download pretrain_encoder.pth now!')
    print('   Notebook → Output → Files → download')

## 8. Quick visual check — noisy vs denoised
Runs the best model on one val slice and plots the result.

In [ ]:
import torch, numpy as np, sys
sys.path.insert(0, PROJECT_ROOT)

from pretraining.unet_denoiser import UNetDenoiser
from pretraining.sts_kernel    import STSModule
from evaluation.visualize      import plot_denoising_comparison

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model
with open(updated_cfg_path) as f:
    cfg = yaml.safe_load(f)

model = UNetDenoiser(
    N=cfg['sts']['N'], in_channels=4, feat_ch=21,
    base_ch=cfg['model']['base_channels'], out_channels=4
).to(device)

sts = STSModule(
    N=cfg['sts']['N'], L=cfg['sts']['L'], eta=cfg['sts']['eta'],
    replace_ratio=cfg['sts']['replace_ratio'],
    window=cfg['sts']['window'], alpha=cfg['sts']['alpha']
).to(device)

ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()

# Load one val window
val_dir   = f'{SLICES_OUT}/val'
val_files = sorted(os.listdir(val_dir))
N = cfg['sts']['N']
half = N // 2

window = torch.stack([
    torch.from_numpy(np.load(f'{val_dir}/{val_files[half + i]}')).float()
    for i in range(-half, half + 1)
], dim=0).unsqueeze(0).to(device)   # (1, N, 4, H, W)

# Add noise
noisy  = (window + torch.randn_like(window) * 0.03).clamp(0, 1)

with torch.no_grad():
    feats    = model.extract_all_features(noisy)
    sampled  = sts(feats, noisy, epoch=29, max_epoch=30)
    denoised = model(sampled)

plot_denoising_comparison(
    noisy=noisy[0, half].cpu(),
    denoised=denoised[0].cpu(),
    clean=window[0, half].cpu(),
    title='Pretraining Visual Check'
)